pipeline lakeflow

In [0]:
import dlt

from pyspark.sql.functions import col, current_timestamp


@dlt.table(
    name="DimProducts_stage"
)
def dimProductsStage():
    df = (
        spark.readStream
        .option("skipChangeCommits", "true")
        .table("databrick_cata.silver.products_silver")
        .filter(col("product_id").isNotNull())
        .withColumn("update_date", current_timestamp())
    )

    return df


@dlt.view(
    name="DimProducts_view"
)
def dimProductsView():
    return dlt.read_stream("DimProducts_stage")


dlt.create_streaming_table(
    name="DimProducts"
)


dlt.apply_changes(
    target="DimProducts",
    source="DimProducts_view",
    keys=["product_id"],
    sequence_by=col("update_date"),
    stored_as_scd_type=1
)